# Gaussian Processes: From Theory to Application
## A Function-Space View of Bayesian Nonparametric Regression

This notebook provides a self-contained introduction to **Gaussian Processes (GPs)** -- a powerful framework for Bayesian nonparametric regression that places distributions directly over functions rather than finite-dimensional parameter vectors.

**What you will learn:**
1. How multivariate Gaussian conditioning gives rise to GP inference
2. The definition and intuition behind Gaussian processes
3. Kernel design and its effect on function smoothness
4. GP regression derived from first principles and implemented via Cholesky
5. Hyperparameter optimization via marginal likelihood
6. Numerical stability considerations
7. Application: uncertainty-aware terrain mapping for robotics

**Prerequisites:** Multivariate Gaussian distributions, linear algebra (Cholesky decomposition), Bayesian inference basics.

**References:**
- Rasmussen & Williams, *Gaussian Processes for Machine Learning*, MIT Press, 2006.
- Murphy, *Machine Learning: A Probabilistic Perspective*, MIT Press, 2012, Chapter 15.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize, approx_fprime
from scipy.spatial.distance import cdist
from scipy.interpolate import RegularGridInterpolator
from scipy.stats import norm

%matplotlib inline

plt.rcParams.update({
    'figure.figsize': (12, 5),
    'font.size': 12,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'lines.linewidth': 2,
})

np.random.seed(42)

In [ ]:
# =============================================================================
# Global Constants
# =============================================================================

SEED = 42
N_PRIOR_SAMPLES = 5
N_GRID = 200
X_MIN, X_MAX = -5.0, 5.0

DEFAULT_LENGTH_SCALE = 1.0
DEFAULT_SIGNAL_VARIANCE = 1.0
DEFAULT_NOISE_VARIANCE = 1e-6

JITTER = 1e-8
CHOLESKY_MAX_JITTER = 1e-2

FD_EPSILON = 1e-5
OPTIM_MAX_ITER = 200

TERRAIN_GRID_SIZE = 50
N_TERRAIN_OBS = 30

INTERP_TOL = 1e-10
GRAD_TOL = 1e-4
VARIANCE_TOL = 1e-8

---
## 1. Introduction: The Function-Space View of Learning

In parametric regression, we fix a model family $f(x; \theta)$ and learn parameters $\theta$. **Gaussian Processes** take a fundamentally different approach: instead of parameterizing functions, we place a **probability distribution directly over the space of functions**.

| Approach | Prior | Posterior |
|----------|-------|----------|
| **Parametric** | $p(\theta)$ over parameters | $p(\theta \mid \mathcal{D})$ |
| **GP (function-space)** | $p(f)$ over functions | $p(f \mid \mathcal{D})$ |

The GP framework is:
- **Nonparametric:** The model complexity grows with the data
- **Bayesian:** We get full posterior distributions, not just point estimates
- **Kernel-based:** All assumptions about smoothness, periodicity, etc. are encoded in the kernel function

The key mathematical insight is that **Gaussian conditioning** -- the formula for $p(x_1 \mid x_2)$ in a joint Gaussian -- is all we need.

---
## 2. Multivariate Gaussian Review

### Conditioning Formula

For a joint Gaussian $\mathbf{x} = (\mathbf{x}_1, \mathbf{x}_2)^T$:

$$\boxed{\mathbf{x}_1 \mid \mathbf{x}_2 \sim \mathcal{N}\left( \boldsymbol{\mu}_1 + \Sigma_{12} \Sigma_{22}^{-1}(\mathbf{x}_2 - \boldsymbol{\mu}_2), \; \Sigma_{11} - \Sigma_{12} \Sigma_{22}^{-1} \Sigma_{21} \right)}$$

### Schur Complement

The conditional covariance $\Sigma_{11} - \Sigma_{12} \Sigma_{22}^{-1} \Sigma_{21}$ is the **Schur complement**. Key properties:
1. Conditional mean is **linear** in $\mathbf{x}_2$
2. Conditional covariance is **independent** of observed value
3. Schur complement is always PSD

These operations -- **joint, conditioning, marginalization** -- are the complete GP toolkit.

In [ ]:
def gaussian_conditional(mu, Sigma, idx1, idx2, x2_observed):
    """Compute conditional p(x1|x2) for a joint Gaussian.

    Args:
        mu: Joint mean vector. Shape: (D,).
        Sigma: Joint covariance matrix. Shape: (D, D).
        idx1: Indices of unknowns. List.
        idx2: Indices of observed. List.
        x2_observed: Observed values. Shape: (len(idx2),).

    Returns:
        mu_cond: Conditional mean. Shape: (len(idx1),).
        Sigma_cond: Conditional covariance. Shape: (len(idx1), len(idx1)).
    """
    mu1, mu2 = mu[idx1], mu[idx2]
    S11 = Sigma[np.ix_(idx1, idx1)]
    S12 = Sigma[np.ix_(idx1, idx2)]
    S22 = Sigma[np.ix_(idx2, idx2)]
    L22 = np.linalg.cholesky(S22 + JITTER * np.eye(len(idx2)))
    alpha = np.linalg.solve(L22.T, np.linalg.solve(L22, x2_observed - mu2))
    mu_cond = mu1 + S12 @ alpha
    V = np.linalg.solve(L22, S12.T)
    Sigma_cond = S11 - V.T @ V
    return mu_cond, Sigma_cond

In [ ]:
# ---- Visualize 2D Gaussian Conditioning ----
mu_joint = np.array([1.0, 2.0])
Sigma_joint = np.array([[1.0, 0.7], [0.7, 1.0]])
x2_obs = np.array([2.5])
mu_cond, Sigma_cond = gaussian_conditional(mu_joint, Sigma_joint, [0], [1], x2_obs)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ax = axes[0]
xg1, xg2 = np.linspace(-2, 4, 200), np.linspace(-1, 5, 200)
X1, X2 = np.meshgrid(xg1, xg2)
pos = np.dstack((X1, X2))
Si = np.linalg.inv(Sigma_joint)
diff = pos - mu_joint
Z = np.exp(-0.5 * np.einsum('...i,ij,...j', diff, Si, diff))
Z = Z / (2 * np.pi * np.sqrt(np.linalg.det(Sigma_joint)))
ax.contourf(X1, X2, Z, levels=20, cmap='Blues', alpha=0.7)
ax.axhline(2.5, color='coral', ls='--', lw=2, label='$x_2=2.5$ observed')
ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
ax.set_title('Joint Gaussian $p(x_1, x_2)$'); ax.legend(fontsize=9)

ax = axes[1]
xf = np.linspace(-2, 4, 300)
mp = np.exp(-0.5*(xf-mu_joint[0])**2/Sigma_joint[0,0]) / np.sqrt(2*np.pi*Sigma_joint[0,0])
cp = np.exp(-0.5*(xf-mu_cond[0])**2/Sigma_cond[0,0]) / np.sqrt(2*np.pi*Sigma_cond[0,0])
ax.plot(xf, mp, color='steelblue', lw=2, label='Marginal $p(x_1)$')
ax.plot(xf, cp, color='coral', lw=2, label='Conditional $p(x_1|x_2=2.5)$')
ax.fill_between(xf, cp, alpha=0.2, color='coral')
ax.set_xlabel('$x_1$'); ax.set_ylabel('Density')
ax.set_title('Marginal vs Conditional'); ax.legend(fontsize=9)
plt.tight_layout(); plt.show()
print(f'Conditional mean: {mu_cond[0]:.4f}, Conditional var: {Sigma_cond[0,0]:.4f}')

---
## 3. Gaussian Process Definition

A **Gaussian process** is a collection of random variables, any finite number of which have a joint Gaussian distribution:

$$f(x) \sim \mathcal{GP}\big(m(x),\; k(x, x')\big)$$

For any finite set of points $\{x_1, \ldots, x_n\}$, the function values are jointly Gaussian. We sample functions by computing $K = k(X_*, X_*)$ and drawing from $\mathcal{N}(\mathbf{0}, K)$.

---
## 4. Kernel Design

**RBF:** $k(x,x') = \sigma^2 \exp(-\|x-x'\|^2 / 2\ell^2)$ -- infinitely smooth

**Matern** ($\nu = 1/2, 3/2, 5/2$): Controls differentiability
- $\nu=1/2$: continuous, not differentiable (Ornstein-Uhlenbeck)
- $\nu=3/2$: once differentiable
- $\nu=5/2$: twice differentiable

**Periodic:** $k(x,x') = \sigma^2 \exp(-2\sin^2(\pi|x-x'|/p) / \ell^2)$

**Linear:** $k(x,x') = \sigma_b^2 + \sigma_v^2(x-c)(x'-c)$

**Kernel algebra:** Sum and product of valid kernels are valid kernels.

In [ ]:
# =============================================================================
# Kernel Functions
# =============================================================================

def rbf_kernel(X1, X2, length_scale=DEFAULT_LENGTH_SCALE, signal_variance=DEFAULT_SIGNAL_VARIANCE):
    """Squared Exponential (RBF) kernel.

    Args:
        X1: First inputs. Shape: (N1,) or (N1, D).
        X2: Second inputs. Shape: (N2,) or (N2, D).
        length_scale: Length scale ell. Scalar.
        signal_variance: Signal variance sigma^2. Scalar.

    Returns:
        Kernel matrix. Shape: (N1, N2).
    """
    X1 = np.atleast_2d(X1).T if X1.ndim == 1 else X1
    X2 = np.atleast_2d(X2).T if X2.ndim == 1 else X2
    dists_sq = cdist(X1, X2, metric='sqeuclidean')
    return signal_variance * np.exp(-0.5 * dists_sq / length_scale**2)


def matern_kernel(X1, X2, length_scale=DEFAULT_LENGTH_SCALE, signal_variance=DEFAULT_SIGNAL_VARIANCE, nu=2.5):
    """Matern kernel for nu in {0.5, 1.5, 2.5}.

    Args:
        X1: First inputs. Shape: (N1,) or (N1, D).
        X2: Second inputs. Shape: (N2,) or (N2, D).
        length_scale: Length scale. Scalar.
        signal_variance: Signal variance. Scalar.
        nu: Smoothness. One of {0.5, 1.5, 2.5}.

    Returns:
        Kernel matrix. Shape: (N1, N2).
    """
    X1 = np.atleast_2d(X1).T if X1.ndim == 1 else X1
    X2 = np.atleast_2d(X2).T if X2.ndim == 1 else X2
    dists = cdist(X1, X2, metric='euclidean')
    r = dists / length_scale
    if nu == 0.5:
        K = np.exp(-r)
    elif nu == 1.5:
        s3r = np.sqrt(3.0) * r
        K = (1.0 + s3r) * np.exp(-s3r)
    elif nu == 2.5:
        s5r = np.sqrt(5.0) * r
        K = (1.0 + s5r + 5.0 * r**2 / 3.0) * np.exp(-s5r)
    else:
        raise ValueError(f'nu must be 0.5, 1.5, or 2.5')
    return signal_variance * K


def periodic_kernel(X1, X2, length_scale=DEFAULT_LENGTH_SCALE, signal_variance=DEFAULT_SIGNAL_VARIANCE, period=1.0):
    """Periodic kernel.

    Args:
        X1: First inputs. Shape: (N1,).
        X2: Second inputs. Shape: (N2,).
        length_scale: Length scale. Scalar.
        signal_variance: Signal variance. Scalar.
        period: Period p. Scalar.

    Returns:
        Kernel matrix. Shape: (N1, N2).
    """
    X1 = np.atleast_2d(X1).T if X1.ndim == 1 else X1
    X2 = np.atleast_2d(X2).T if X2.ndim == 1 else X2
    dists = cdist(X1, X2, metric='euclidean')
    return signal_variance * np.exp(-2.0 * np.sin(np.pi * dists / period)**2 / length_scale**2)


def linear_kernel(X1, X2, signal_variance=DEFAULT_SIGNAL_VARIANCE, bias_variance=0.1, center=0.0):
    """Linear kernel.

    Args:
        X1: First inputs. Shape: (N1,).
        X2: Second inputs. Shape: (N2,).
        signal_variance: Slope variance. Scalar.
        bias_variance: Bias variance. Scalar.
        center: Center point. Scalar.

    Returns:
        Kernel matrix. Shape: (N1, N2).
    """
    X1 = np.atleast_2d(X1).T if X1.ndim == 1 else X1
    X2 = np.atleast_2d(X2).T if X2.ndim == 1 else X2
    return bias_variance + signal_variance * (X1 - center) @ (X2 - center).T

In [ ]:
# =============================================================================
# Core GP Utilities
# =============================================================================

def safe_cholesky(K, max_jitter=CHOLESKY_MAX_JITTER):
    """Cholesky with adaptive jitter.

    Args:
        K: PSD matrix. Shape: (N, N).
        max_jitter: Max jitter. Scalar.

    Returns:
        L: Lower Cholesky factor. Shape: (N, N).
    """
    n = K.shape[0]
    jitter = JITTER
    while jitter <= max_jitter:
        try:
            return np.linalg.cholesky(K + jitter * np.eye(n))
        except np.linalg.LinAlgError:
            jitter *= 10
    raise np.linalg.LinAlgError(f'Cholesky failed with jitter={max_jitter:.1e}')


def gp_sample_prior(X, kernel_fn, n_samples=N_PRIOR_SAMPLES, **kw):
    """Draw function samples from GP prior.

    Args:
        X: Input points. Shape: (N,).
        kernel_fn: Kernel function.
        n_samples: Number of samples. Integer.

    Returns:
        samples: Shape: (n_samples, N).
    """
    K = kernel_fn(X, X, **kw)
    L = safe_cholesky(K)
    return (L @ np.random.randn(K.shape[0], n_samples)).T

In [ ]:
# ---- 4-Panel: Kernel comparison via GP prior samples ----
np.random.seed(SEED)
X_grid = np.linspace(X_MIN, X_MAX, N_GRID)
kernels = [('RBF (SE)', rbf_kernel, {}), ('Matern 1/2', matern_kernel, {'nu': 0.5}),
           ('Matern 3/2', matern_kernel, {'nu': 1.5}), ('Matern 5/2', matern_kernel, {'nu': 2.5})]
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
sc = ['steelblue', 'coral', 'seagreen', 'goldenrod', 'mediumpurple']
for ax, (name, kfn, kw) in zip(axes.ravel(), kernels):
    samples = gp_sample_prior(X_grid, kfn, n_samples=N_PRIOR_SAMPLES, **kw)
    for i, s in enumerate(samples):
        ax.plot(X_grid, s, color=sc[i], alpha=0.8)
    ax.set_title(f'{name} -- prior samples')
    ax.set_xlabel('$x$'); ax.set_ylabel('$f(x)$')
plt.tight_layout(); plt.show()

In [ ]:
# ---- Kernel profiles k(0, x) ----
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
xt = np.linspace(-4, 4, 300)
x0 = np.array([0.0])
ax = axes[0]
for nm,kf,kw,col in [('RBF',rbf_kernel,{},'steelblue'),('Matern 1/2',matern_kernel,{'nu':0.5},'coral'),
                       ('Matern 3/2',matern_kernel,{'nu':1.5},'seagreen'),('Matern 5/2',matern_kernel,{'nu':2.5},'goldenrod')]:
    ax.plot(xt, kf(x0,xt,**kw).ravel(), color=col, lw=2, label=nm)
ax.set_xlabel("$x'$"); ax.set_ylabel("$k(0,x')$"); ax.set_title('Kernel Profiles'); ax.legend()
ax = axes[1]
for ell,col in [(0.3,'steelblue'),(1.0,'coral'),(3.0,'seagreen')]:
    ax.plot(xt, rbf_kernel(x0,xt,length_scale=ell).ravel(), color=col, lw=2, label=f'ell={ell}')
ax.set_xlabel("$x'$"); ax.set_ylabel("$k(0,x')$"); ax.set_title('RBF: Length Scale Effect'); ax.legend()
plt.tight_layout(); plt.show()

---
## 5. GP Regression

### Derivation via Gaussian Conditioning

$$\boxed{f_* \mid X, \mathbf{y}, X_* \sim \mathcal{N}(\bar{f}_*, \text{cov}(f_*))}$$

where:
$$\bar{f}_* = K(X_*, X) [K(X, X) + \sigma_n^2 I]^{-1} \mathbf{y}$$
$$\text{cov}(f_*) = K(X_*, X_*) - K(X_*, X) [K(X, X) + \sigma_n^2 I]^{-1} K(X, X_*)$$

### Cholesky Implementation
1. $L = \text{chol}(K_y)$
2. Solve $L^T \boldsymbol{\alpha} = L^{-1}\mathbf{y}$
3. Mean: $\bar{f}_* = K_{*n} \boldsymbol{\alpha}$
4. Variance: $V = L^{-1} K_{n*}$, then $\text{cov} = K_{**} - V^T V$

In [ ]:
# =============================================================================
# GP Posterior Inference
# =============================================================================

def gp_posterior(X_train, y_train, X_test, kernel_fn, noise_variance=DEFAULT_NOISE_VARIANCE, **kernel_kwargs):
    """Compute the GP posterior predictive distribution.

    Args:
        X_train: Training inputs. Shape: (N,) or (N, D).
        y_train: Training targets. Shape: (N,).
        X_test: Test inputs. Shape: (M,) or (M, D).
        kernel_fn: Kernel function.
        noise_variance: Observation noise variance. Scalar.

    Returns:
        mu_post: Posterior mean. Shape: (M,).
        var_post: Posterior variance. Shape: (M,).
        cov_post: Posterior covariance. Shape: (M, M).
    """
    N = len(X_train)
    K_nn = kernel_fn(X_train, X_train, **kernel_kwargs)
    K_nm = kernel_fn(X_train, X_test, **kernel_kwargs)
    K_mm = kernel_fn(X_test, X_test, **kernel_kwargs)
    K_y = K_nn + noise_variance * np.eye(N)
    L = safe_cholesky(K_y)
    alpha = np.linalg.solve(L.T, np.linalg.solve(L, y_train))
    mu_post = K_nm.T @ alpha
    V = np.linalg.solve(L, K_nm)
    cov_post = K_mm - V.T @ V
    var_post = np.diag(cov_post)
    return mu_post, var_post, cov_post


def gp_sample_posterior(X_train, y_train, X_test, kernel_fn, n_samples=N_PRIOR_SAMPLES,
                        noise_variance=DEFAULT_NOISE_VARIANCE, **kernel_kwargs):
    """Draw function samples from the GP posterior.

    Args:
        X_train: Training inputs. Shape: (N,).
        y_train: Training targets. Shape: (N,).
        X_test: Test inputs. Shape: (M,).
        kernel_fn: Kernel function.
        n_samples: Number of samples. Integer.
        noise_variance: Noise variance. Scalar.

    Returns:
        samples: Shape: (n_samples, M).
    """
    mu_post, _, cov_post = gp_posterior(X_train, y_train, X_test, kernel_fn,
                                         noise_variance=noise_variance, **kernel_kwargs)
    L_post = safe_cholesky(cov_post)
    M = len(X_test)
    return (mu_post[:, None] + L_post @ np.random.randn(M, n_samples)).T

In [ ]:
# ---- GP Regression Demo ----
np.random.seed(SEED)
N_TRAIN = 8
X_train = np.sort(np.random.uniform(X_MIN + 1, X_MAX - 1, N_TRAIN))
y_train = np.sin(X_train)
X_test = np.linspace(X_MIN, X_MAX, N_GRID)

mu_post, var_post, _ = gp_posterior(X_train, y_train, X_test, rbf_kernel,
    noise_variance=1e-10, length_scale=1.0, signal_variance=1.0)
posterior_samples = gp_sample_posterior(X_train, y_train, X_test, rbf_kernel,
    n_samples=5, noise_variance=1e-10, length_scale=1.0, signal_variance=1.0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ax = axes[0]
np.random.seed(SEED)
prior_samples = gp_sample_prior(X_test, rbf_kernel, n_samples=5, length_scale=1.0)
for s in prior_samples: ax.plot(X_test, s, alpha=0.6)
ax.fill_between(X_test, -2, 2, alpha=0.1, color='steelblue', label='$\\pm 2\\sigma$ prior')
ax.set_xlabel('$x$'); ax.set_ylabel('$f(x)$'); ax.set_title('GP Prior'); ax.set_ylim(-3, 3); ax.legend()

ax = axes[1]
std_post = np.sqrt(np.maximum(var_post, 0))
ax.fill_between(X_test, mu_post - 2*std_post, mu_post + 2*std_post, alpha=0.2, color='steelblue', label='$\\pm 2\\sigma$')
ax.plot(X_test, mu_post, color='steelblue', lw=2, label='Posterior mean')
for s in posterior_samples: ax.plot(X_test, s, alpha=0.4, lw=1)
ax.plot(X_train, y_train, 'ko', ms=8, zorder=5, label='Observations')
ax.plot(X_test, np.sin(X_test), 'k--', alpha=0.5, label='True sin(x)')
ax.set_xlabel('$x$'); ax.set_ylabel('$f(x)$'); ax.set_title('GP Posterior'); ax.set_ylim(-3, 3); ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
# =============================================================================
# Verification: Posterior Properties (Noiseless)
# =============================================================================

mu_at_train, var_at_train, _ = gp_posterior(
    X_train, y_train, X_train, rbf_kernel,
    noise_variance=1e-10, length_scale=1.0, signal_variance=1.0)

interp_residual = np.max(np.abs(mu_at_train - y_train))
status_interp = 'PASS' if interp_residual < INTERP_TOL else 'FAIL'
print(f'Interpolation: max residual |mu(x_i) - y_i| = {interp_residual:.2e} [{status_interp}]')

max_var_at_train = np.max(np.abs(var_at_train))
status_var = 'PASS' if max_var_at_train < VARIANCE_TOL else 'FAIL'
print(f'Posterior variance at obs: max var = {max_var_at_train:.2e} [{status_var}]')

min_var_away = np.min(var_post)
status_pos = 'PASS' if min_var_away >= -VARIANCE_TOL else 'FAIL'
print(f'Posterior variance non-negative: min var = {min_var_away:.2e} [{status_pos}]')

---
## 6. Hyperparameter Optimization

### Log Marginal Likelihood

$$\boxed{\log p(\mathbf{y} \mid X, \theta) = -\frac{1}{2}\mathbf{y}^T K_y^{-1} \mathbf{y} - \frac{1}{2}\log|K_y| - \frac{n}{2}\log 2\pi}$$

### Gradient

$$\frac{\partial}{\partial \theta_j} \log p(\mathbf{y} \mid X, \theta) = \frac{1}{2}\text{tr}\left[(\boldsymbol{\alpha}\boldsymbol{\alpha}^T - K_y^{-1}) \frac{\partial K_y}{\partial \theta_j}\right]$$

In [ ]:
def log_marginal_likelihood(X_train, y_train, kernel_fn, noise_variance, **kernel_kwargs):
    """Compute log marginal likelihood.

    Args:
        X_train: Training inputs. Shape: (N,).
        y_train: Training targets. Shape: (N,).
        kernel_fn: Kernel function.
        noise_variance: Noise variance. Scalar.

    Returns:
        lml: Log marginal likelihood. Scalar.
    """
    N = len(X_train)
    K = kernel_fn(X_train, X_train, **kernel_kwargs)
    K_y = K + noise_variance * np.eye(N)
    L = safe_cholesky(K_y)
    alpha = np.linalg.solve(L.T, np.linalg.solve(L, y_train))
    data_fit = -0.5 * y_train @ alpha
    complexity = -np.sum(np.log(np.diag(L)))
    normalization = -0.5 * N * np.log(2 * np.pi)
    return data_fit + complexity + normalization


def optimize_hyperparameters(X_train, y_train, kernel_fn, theta0, param_names, bounds=None, max_iter=OPTIM_MAX_ITER):
    """Optimize GP hyperparameters via log marginal likelihood.

    Args:
        X_train: Training inputs. Shape: (N,).
        y_train: Training targets. Shape: (N,).
        kernel_fn: Kernel function.
        theta0: Initial log-hyperparameters. Shape: (P,).
        param_names: Kernel parameter names. List.
        bounds: Bounds in log-space. List or None.
        max_iter: Max iterations. Integer.

    Returns:
        theta_opt: Optimized params (original space). Dict.
        lml_opt: Optimal LML. Scalar.
        result: Scipy result object.
    """
    def neg_lml(log_theta):
        nv = np.exp(log_theta[0])
        kw = {name: np.exp(log_theta[i+1]) for i, name in enumerate(param_names)}
        return -log_marginal_likelihood(X_train, y_train, kernel_fn, nv, **kw)
    if bounds is None:
        bounds = [(-10, 10)] * len(theta0)
    result = minimize(neg_lml, theta0, method='L-BFGS-B', bounds=bounds, options={'maxiter': max_iter})
    theta_opt = {'noise_variance': np.exp(result.x[0])}
    for i, name in enumerate(param_names):
        theta_opt[name] = np.exp(result.x[i + 1])
    return theta_opt, -result.fun, result

In [ ]:
# ---- Hyperparameter Optimization Demo ----
np.random.seed(SEED)
N_TRAIN_NOISY = 20
X_train_noisy = np.sort(np.random.uniform(X_MIN + 1, X_MAX - 1, N_TRAIN_NOISY))
y_train_noisy = np.sin(X_train_noisy) + 0.2 * np.random.randn(N_TRAIN_NOISY)

theta0 = np.log([0.1, 0.5, 0.5])
theta_opt, lml_opt, opt_result = optimize_hyperparameters(
    X_train_noisy, y_train_noisy, rbf_kernel, theta0, param_names=['length_scale', 'signal_variance'])

print('Optimized hyperparameters:')
for name, val in theta_opt.items():
    print(f'  {name:20s} = {val:.6f}')
print(f'Log marginal likelihood: {lml_opt:.4f}')

X_test_dense = np.linspace(X_MIN, X_MAX, N_GRID)
mu_opt, var_opt, _ = gp_posterior(X_train_noisy, y_train_noisy, X_test_dense, rbf_kernel, **theta_opt)
mu_bad, var_bad, _ = gp_posterior(X_train_noisy, y_train_noisy, X_test_dense, rbf_kernel,
    noise_variance=0.01, length_scale=0.2, signal_variance=2.0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ax = axes[0]
std_o = np.sqrt(np.maximum(var_opt, 0))
ax.fill_between(X_test_dense, mu_opt-2*std_o, mu_opt+2*std_o, alpha=0.2, color='steelblue')
ax.plot(X_test_dense, mu_opt, color='steelblue', lw=2, label='Posterior mean')
ax.plot(X_train_noisy, y_train_noisy, 'ko', ms=6); ax.plot(X_test_dense, np.sin(X_test_dense), 'k--', alpha=0.5)
ax.set_title('Optimized hyperparameters'); ax.set_xlabel('$x$'); ax.legend(fontsize=9)
ax = axes[1]
std_b = np.sqrt(np.maximum(var_bad, 0))
ax.fill_between(X_test_dense, mu_bad-2*std_b, mu_bad+2*std_b, alpha=0.2, color='coral')
ax.plot(X_test_dense, mu_bad, color='coral', lw=2, label='Posterior mean')
ax.plot(X_train_noisy, y_train_noisy, 'ko', ms=6); ax.plot(X_test_dense, np.sin(X_test_dense), 'k--', alpha=0.5)
ax.set_title('Poor hyperparameters'); ax.set_xlabel('$x$'); ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
# =============================================================================
# Verification: Marginal Likelihood Gradient
# =============================================================================

def lml_for_params(log_params, X_tr, y_tr):
    """LML from flat log-hyperparameter vector.

    Args:
        log_params: [log(noise_var), log(ell), log(sig_var)]. Shape: (3,).
        X_tr: Training inputs. Shape: (N,).
        y_tr: Training targets. Shape: (N,).

    Returns:
        Log marginal likelihood. Scalar.
    """
    return log_marginal_likelihood(X_tr, y_tr, rbf_kernel, np.exp(log_params[0]),
                                   length_scale=np.exp(log_params[1]), signal_variance=np.exp(log_params[2]))

log_theta_test = np.log([0.05, 1.0, 1.0])
num_grad = np.zeros(3)
for i in range(3):
    e = np.zeros(3); e[i] = FD_EPSILON
    num_grad[i] = (lml_for_params(log_theta_test + e, X_train_noisy, y_train_noisy)
                   - lml_for_params(log_theta_test - e, X_train_noisy, y_train_noisy)) / (2 * FD_EPSILON)
scipy_grad = approx_fprime(log_theta_test, lambda t: lml_for_params(t, X_train_noisy, y_train_noisy), FD_EPSILON)
grad_error = np.max(np.abs(num_grad - scipy_grad) / (np.abs(num_grad) + 1e-12))
status_grad = 'PASS' if grad_error < GRAD_TOL else 'FAIL'
print(f'Marginal likelihood gradient: max relative error = {grad_error:.2e} [{status_grad}]')

In [ ]:
# ---- Marginal Likelihood Landscape ----
n_grid_lml = 40
ell_range = np.linspace(0.2, 4.0, n_grid_lml)
sig2_range = np.linspace(0.1, 3.0, n_grid_lml)
ELL_G, SIG2_G = np.meshgrid(ell_range, sig2_range)
LML_G = np.zeros_like(ELL_G)
nv_fixed = theta_opt['noise_variance']
for i in range(n_grid_lml):
    for j in range(n_grid_lml):
        LML_G[i,j] = log_marginal_likelihood(X_train_noisy, y_train_noisy, rbf_kernel, nv_fixed,
            length_scale=ELL_G[i,j], signal_variance=SIG2_G[i,j])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ax = axes[0]
cs = ax.contourf(ELL_G, SIG2_G, LML_G, levels=30, cmap='viridis')
ax.plot(theta_opt['length_scale'], theta_opt['signal_variance'], 'r*', ms=15, label='Optimum')
ax.set_xlabel('Length scale $\\ell$'); ax.set_ylabel('Signal variance $\\sigma^2$')
ax.set_title('Log Marginal Likelihood Landscape'); ax.legend(); fig.colorbar(cs, ax=ax)
ax = axes[1]
lml_sl = [log_marginal_likelihood(X_train_noisy, y_train_noisy, rbf_kernel, nv_fixed,
    length_scale=e, signal_variance=theta_opt['signal_variance']) for e in ell_range]
ax.plot(ell_range, lml_sl, color='steelblue', lw=2)
ax.axvline(theta_opt['length_scale'], color='coral', ls='--', lw=2, label=f'Optimal ell={theta_opt["length_scale"]:.2f}')
ax.set_xlabel('$\\ell$'); ax.set_ylabel('LML'); ax.set_title('LML Slice'); ax.legend()
plt.tight_layout(); plt.show()

---
## 7. Numerical Stability

### Jitter
Add $\epsilon I$ to $K_y$. Our `safe_cholesky` does this adaptively.

### Condition Number
$\kappa(K_y) = \lambda_{\max}/\lambda_{\min}$. We lose $\log_{10}(\kappa)$ digits.

### Complexity
| Operation | Cost |
|-----------|------|
| Cholesky | $O(n^3/3)$ |
| Training total | $O(n^3)$ |
| Prediction ($m$ pts) | $O(n^2 m)$ |

In [ ]:
# =============================================================================
# Verification: Cholesky + Jitter
# =============================================================================

X_close = np.linspace(0, 1, 50)
K_tricky = rbf_kernel(X_close, X_close, length_scale=0.01, signal_variance=1.0)
try:
    np.linalg.cholesky(K_tricky)
    chol_raw = 'succeeded'
except np.linalg.LinAlgError:
    chol_raw = 'FAILED'
try:
    safe_cholesky(K_tricky)
    chol_safe = 'PASS'
except np.linalg.LinAlgError:
    chol_safe = 'FAIL'
print(f'Cholesky without jitter: {chol_raw}')
print(f'Cholesky with safe_cholesky: [{chol_safe}]')

print('\nCondition number analysis:')
for ell in [0.01, 0.1, 1.0, 10.0]:
    K_c = rbf_kernel(X_close, X_close, length_scale=ell)
    cond = np.linalg.cond(K_c)
    print(f'  ell={ell:5.2f}: kappa = {cond:.2e}, digits lost ~ {np.log10(max(cond,1.0)):.1f}')

In [ ]:
# ---- Log-determinant: Cholesky vs slogdet ----
sizes = [10, 50, 100, 200, 500]
print(f'{"N":>6s}  {"Cholesky":>12s}  {"slogdet":>12s}  {"Rel Error":>12s}')
for n in sizes:
    X_n = np.linspace(0, 5, n)
    K_n = rbf_kernel(X_n, X_n, length_scale=1.0) + 1e-6 * np.eye(n)
    L_n = np.linalg.cholesky(K_n)
    logdet_chol = 2 * np.sum(np.log(np.diag(L_n)))
    _, logdet_slog = np.linalg.slogdet(K_n)
    rel_err = abs(logdet_chol - logdet_slog) / (abs(logdet_slog) + 1e-12)
    print(f'{n:6d}  {logdet_chol:12.4f}  {logdet_slog:12.4f}  {rel_err:12.2e}')

---
## 8. Application: Terrain Mapping

A robot traverses unknown terrain collecting sparse elevation measurements. We use GP regression to reconstruct the terrain surface and quantify uncertainty for path planning.

In [ ]:
# ---- Terrain Mapping ----
np.random.seed(SEED)
x1t = np.linspace(0, 10, TERRAIN_GRID_SIZE)
x2t = np.linspace(0, 10, TERRAIN_GRID_SIZE)
X1_T, X2_T = np.meshgrid(x1t, x2t)
X_full = np.column_stack([X1_T.ravel(), X2_T.ravel()])
K_true = rbf_kernel(X_full, X_full, length_scale=2.0, signal_variance=1.0)
L_true = safe_cholesky(K_true)
z_true = L_true @ np.random.randn(len(X_full))
Z_true = z_true.reshape(TERRAIN_GRID_SIZE, TERRAIN_GRID_SIZE)

t_path = np.linspace(0, 1, 15)
px1 = 1.0 + 8.0 * t_path
px2 = 5.0 + 3.0 * np.sin(2 * np.pi * t_path)
rx1 = np.random.uniform(0.5, 9.5, N_TERRAIN_OBS - 15)
rx2 = np.random.uniform(0.5, 9.5, N_TERRAIN_OBS - 15)
ox1 = np.concatenate([px1, rx1])
ox2 = np.concatenate([px2, rx2])
X_obs = np.column_stack([ox1, ox2])
terp = RegularGridInterpolator((x1t, x2t), Z_true.T)
y_obs = terp(X_obs) + 0.05 * np.random.randn(len(X_obs))
print(f'Terrain: {TERRAIN_GRID_SIZE}x{TERRAIN_GRID_SIZE}, Observations: {len(X_obs)}')

In [ ]:
# ---- GP Terrain Prediction ----
mu_ter, var_ter, _ = gp_posterior(X_obs, y_obs, X_full, rbf_kernel,
    noise_variance=0.05**2, length_scale=2.0, signal_variance=1.0)
MU_T = mu_ter.reshape(TERRAIN_GRID_SIZE, TERRAIN_GRID_SIZE)
STD_T = np.sqrt(np.maximum(var_ter, 0)).reshape(TERRAIN_GRID_SIZE, TERRAIN_GRID_SIZE)

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
ax = axes[0,0]
im = ax.contourf(X1_T, X2_T, Z_true, levels=20, cmap='terrain')
ax.plot(ox1, ox2, 'k.', ms=5); ax.set_title('Ground Truth'); fig.colorbar(im, ax=ax)
ax = axes[0,1]
im = ax.contourf(X1_T, X2_T, MU_T, levels=20, cmap='terrain')
ax.plot(ox1, ox2, 'k.', ms=5); ax.set_title('GP Posterior Mean'); fig.colorbar(im, ax=ax)
ax = axes[1,0]
im = ax.contourf(X1_T, X2_T, STD_T, levels=20, cmap='YlOrRd')
ax.plot(ox1, ox2, 'k.', ms=5); ax.set_title('Uncertainty ($\\sigma$)'); fig.colorbar(im, ax=ax)
ax = axes[1,1]
im = ax.contourf(X1_T, X2_T, np.abs(MU_T - Z_true), levels=20, cmap='Reds')
ax.plot(ox1, ox2, 'k.', ms=5); ax.set_title('Reconstruction Error'); fig.colorbar(im, ax=ax)
for ax in axes.ravel(): ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
plt.tight_layout(); plt.show()
print(f'RMSE: {np.sqrt(np.mean((MU_T - Z_true)**2)):.4f}')

In [ ]:
# ---- Uncertainty-Aware Path Cost ----
LAMBDA_UNCERTAINTY = 2.0
cost_surface = MU_T + LAMBDA_UNCERTAINTY * STD_T
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
ax = axes[0]
im = ax.contourf(X1_T, X2_T, MU_T, levels=20, cmap='terrain')
ax.plot(ox1, ox2, 'k.', ms=4); ax.set_title('Mean Elevation'); fig.colorbar(im, ax=ax)
ax = axes[1]
im = ax.contourf(X1_T, X2_T, STD_T, levels=20, cmap='YlOrRd')
ax.plot(ox1, ox2, 'k.', ms=4); ax.set_title('Uncertainty'); fig.colorbar(im, ax=ax)
ax = axes[2]
im = ax.contourf(X1_T, X2_T, cost_surface, levels=20, cmap='hot_r')
ax.plot(ox1, ox2, 'k.', ms=4); ax.set_title('Path Cost: $\\mu + 2\\sigma$'); fig.colorbar(im, ax=ax)
for ax in axes: ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
plt.tight_layout(); plt.show()

---
## 9. Comprehensive Visualizations

In [ ]:
# ---- GP Posterior Evolution ----
np.random.seed(SEED)
X_all = np.sort(np.random.uniform(X_MIN+1, X_MAX-1, 20))
y_all = np.sin(X_all)
X_vis = np.linspace(X_MIN, X_MAX, N_GRID)
fig, axes = plt.subplots(1, 5, figsize=(24, 4))
for ax, n_obs in zip(axes, [0, 2, 5, 10, 20]):
    if n_obs == 0:
        mu_v, var_v = np.zeros(N_GRID), np.ones(N_GRID)
    else:
        mu_v, var_v, _ = gp_posterior(X_all[:n_obs], y_all[:n_obs], X_vis, rbf_kernel,
            noise_variance=1e-10, length_scale=1.0, signal_variance=1.0)
    std_v = np.sqrt(np.maximum(var_v, 0))
    ax.fill_between(X_vis, mu_v-2*std_v, mu_v+2*std_v, alpha=0.2, color='steelblue')
    ax.plot(X_vis, mu_v, color='steelblue', lw=2)
    ax.plot(X_vis, np.sin(X_vis), 'k--', alpha=0.3)
    if n_obs > 0: ax.plot(X_all[:n_obs], y_all[:n_obs], 'ko', ms=5)
    ax.set_title(f'$n={n_obs}$'); ax.set_ylim(-3, 3); ax.set_xlabel('$x$')
plt.suptitle('GP Posterior Evolution', fontsize=14, y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# ---- Kernel Smoothness Comparison ----
np.random.seed(SEED)
X_tc = np.array([-4.0, -2.5, -1.0, 0.0, 1.5, 3.0, 4.5])
y_tc = np.sin(X_tc)
X_tst = np.linspace(X_MIN, X_MAX, N_GRID)
kconfigs = [('Matern 1/2', matern_kernel, {'nu':0.5}), ('Matern 3/2', matern_kernel, {'nu':1.5}),
            ('Matern 5/2', matern_kernel, {'nu':2.5}), ('RBF', rbf_kernel, {})]
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
kcols = ['coral', 'goldenrod', 'seagreen', 'steelblue']
for ax, (nm, kf, kw), col in zip(axes.ravel(), kconfigs, kcols):
    mu_k, var_k, _ = gp_posterior(X_tc, y_tc, X_tst, kf, noise_variance=1e-10,
        length_scale=1.0, signal_variance=1.0, **kw)
    std_k = np.sqrt(np.maximum(var_k, 0))
    ax.fill_between(X_tst, mu_k-2*std_k, mu_k+2*std_k, alpha=0.15, color=col)
    ax.plot(X_tst, mu_k, color=col, lw=2, label='Mean')
    samps = gp_sample_posterior(X_tc, y_tc, X_tst, kf, n_samples=3, noise_variance=1e-10,
        length_scale=1.0, signal_variance=1.0, **kw)
    for s in samps: ax.plot(X_tst, s, color=col, alpha=0.3, lw=1)
    ax.plot(X_tc, y_tc, 'ko', ms=7, zorder=5)
    ax.set_title(nm); ax.set_ylim(-2.5, 2.5); ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
# ---- Length Scale Effect ----
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, ell in zip(axes, [0.3, 1.0, 3.0]):
    mu_e, var_e, _ = gp_posterior(X_tc, y_tc, X_tst, rbf_kernel,
        noise_variance=1e-10, length_scale=ell, signal_variance=1.0)
    std_e = np.sqrt(np.maximum(var_e, 0))
    ax.fill_between(X_tst, mu_e-2*std_e, mu_e+2*std_e, alpha=0.2, color='steelblue')
    ax.plot(X_tst, mu_e, color='steelblue', lw=2)
    ax.plot(X_tc, y_tc, 'ko', ms=7, zorder=5)
    ax.set_title(f'RBF, ell={ell}'); ax.set_ylim(-2.5, 2.5)
plt.tight_layout(); plt.show()

---
## 10. Extensions

### Sparse GPs
$O(nm^2)$ via $m \ll n$ inducing points. Key: FITC, SVGP.

### GP Classification
Laplace approximation: $p(f|\mathbf{y}) \approx \mathcal{N}(\hat{f}, (K^{-1}+W)^{-1})$

### Bayesian Optimization
Expected Improvement: $\alpha_{EI}(x) = \mathbb{E}[\max(f(x)-f^+, 0)]$

### SDE-GP Correspondence
Matern GPs = SDEs, enabling $O(n)$ Kalman inference.

| Concept | Insight |
|---------|--------|
| **GP** | Distribution over functions via mean + kernel |
| **Kernel** | Encodes smoothness, scale, periodicity |
| **Posterior** | Exact Gaussian conditioning |
| **Hyperparameters** | Marginal likelihood = automatic Occam's razor |
| **Scalability** | $O(n^3)$ exact; sparse for $n>10^3$ |

In [ ]:
# ---- Bayesian Optimization Demo ----
np.random.seed(SEED)

def expected_improvement(mu, sigma, f_best):
    """Expected Improvement acquisition function.

    Args:
        mu: Posterior mean. Shape: (M,).
        sigma: Posterior std. Shape: (M,).
        f_best: Best observed value. Scalar.

    Returns:
        ei: Expected improvement. Shape: (M,).
    """
    sigma_safe = np.maximum(sigma, 1e-12)
    z = (mu - f_best) / sigma_safe
    ei = (mu - f_best) * norm.cdf(z) + sigma_safe * norm.pdf(z)
    ei[sigma < 1e-12] = 0.0
    return ei

objective = lambda x: -(np.sin(3*x)*x + 0.5*np.cos(5*x)*x)
X_bo = np.array([-3.0, -1.0, 1.0, 3.5])
y_bo = objective(X_bo)
X_bo_t = np.linspace(X_MIN, X_MAX, 300)
mu_bo, var_bo, _ = gp_posterior(X_bo, y_bo, X_bo_t, rbf_kernel,
    noise_variance=1e-10, length_scale=1.0, signal_variance=2.0)
std_bo = np.sqrt(np.maximum(var_bo, 0))
ei = expected_improvement(mu_bo, std_bo, np.max(y_bo))
next_x = X_bo_t[np.argmax(ei)]

fig, axes = plt.subplots(2, 1, figsize=(12, 8), gridspec_kw={'height_ratios': [2, 1]})
ax = axes[0]
ax.fill_between(X_bo_t, mu_bo-2*std_bo, mu_bo+2*std_bo, alpha=0.2, color='steelblue')
ax.plot(X_bo_t, mu_bo, color='steelblue', lw=2, label='GP mean')
ax.plot(X_bo_t, objective(X_bo_t), 'k--', alpha=0.5, label='True')
ax.plot(X_bo, y_bo, 'ko', ms=8, zorder=5, label='Obs')
ax.axvline(next_x, color='coral', ls=':', lw=2, label=f'Next query (x={next_x:.2f})')
ax.set_ylabel('$f(x)$'); ax.set_title('Bayesian Optimization'); ax.legend(fontsize=9)
ax = axes[1]
ax.fill_between(X_bo_t, 0, ei, alpha=0.3, color='coral')
ax.plot(X_bo_t, ei, color='coral', lw=2)
ax.axvline(next_x, color='coral', ls=':', lw=2)
ax.set_xlabel('$x$'); ax.set_ylabel('EI$(x)$'); ax.set_title('Expected Improvement')
plt.tight_layout(); plt.show()

In [ ]:
# =============================================================================
# Final Summary: All Verification Tests
# =============================================================================

print('=' * 70)
print('GAUSSIAN PROCESSES - VERIFICATION SUMMARY')
print('=' * 70)
results = []

mu_chk, var_chk, _ = gp_posterior(X_train, y_train, X_train, rbf_kernel,
    noise_variance=1e-10, length_scale=1.0, signal_variance=1.0)
r1 = np.max(np.abs(mu_chk - y_train))
p1 = r1 < INTERP_TOL; results.append(p1)
print(f'  1. Interpolation: max |mu(x_i) - y_i| = {r1:.2e} [{"PASS" if p1 else "FAIL"}]')

r2 = np.max(np.abs(var_chk))
p2 = r2 < VARIANCE_TOL; results.append(p2)
print(f'  2. Posterior variance at obs: max var = {r2:.2e} [{"PASS" if p2 else "FAIL"}]')

lt = np.log([0.05, 1.0, 1.0])
ng = np.zeros(3)
for i in range(3):
    e = np.zeros(3); e[i] = FD_EPSILON
    ng[i] = (lml_for_params(lt+e, X_train_noisy, y_train_noisy) - lml_for_params(lt-e, X_train_noisy, y_train_noisy))/(2*FD_EPSILON)
sg = approx_fprime(lt, lambda t: lml_for_params(t, X_train_noisy, y_train_noisy), FD_EPSILON)
r3 = np.max(np.abs(ng - sg) / (np.abs(ng) + 1e-12))
p3 = r3 < GRAD_TOL; results.append(p3)
print(f'  3. LML gradient: max relative error = {r3:.2e} [{"PASS" if p3 else "FAIL"}]')

try:
    safe_cholesky(K_tricky); p4 = True
except: p4 = False
results.append(p4)
print(f'  4. Cholesky succeeds with jitter: [{"PASS" if p4 else "FAIL"}]')

print('=' * 70)
print(f'  Total: {sum(results)}/{len(results)} tests passed')
print('=' * 70)